In [1]:
import pandas as pd
import plotly.express as px

In [3]:
df = pd.read_csv("data/team_week_points.csv")
df = df[df.manager.notnull()]
len(df)

1260

In [ ]:
#3 — Points For vs. Points Against (Luck Scatter)

# In standings_line.py or a new page
import plotly.graph_objects as go

s = pd.read_csv("data/standings_data.csv")
s = s[s.manager.notnull()]

# Filter to final week of selected season
final = s[s['season'] == selected_year]
final = final[final['week'] == final['week'].max()]

ref = max(final['points_for'].max(), final['points_against'].max())

fig = px.scatter(
    final, x='points_against', y='points_for',
    text='manager', color='standings_rank',
    color_continuous_scale='RdYlGn_r',
    labels={'points_for': 'Points Scored (PS)', 'points_against': 'Points Allowed (PA)'},
    title=f"Luck Chart — {selected_year}"
)
# Diagonal = perfectly even PS/PA
fig.add_shape(type='line', x0=0, y0=0, x1=ref, y1=ref,
              line=dict(color='gray', dash='dash'))
fig.update_traces(textposition='top center', marker=dict(size=12))
fig.update_coloraxes(showscale=False)
st.plotly_chart(fig, use_container_width=True)

In [ ]:
#5 — Radar / Spider Chart (Manager Stat Fingerprint)

import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler  # pip install scikit-learn

h = pd.read_csv("data/hitting_stats.csv")
p = pd.read_csv("data/pitching_stats.csv")

HIT_COLS = ['HR', 'SB', 'R', 'RBI', 'TB']
PIT_COLS = ['K', 'QS', 'K_per_IP']  # keeping ERA/WHIP out since lower=better complicates scaling

selected_year = st.selectbox("Select year", sorted(h['season'].unique(), reverse=True))

h_agg = h[h['season'] == selected_year].groupby('manager')[HIT_COLS].sum()
p_agg = p[p['season'] == selected_year].groupby('manager')[PIT_COLS].sum()
combined = h_agg.join(p_agg)

# Normalize each stat 0–1 across managers
scaler = MinMaxScaler()
scaled = pd.DataFrame(scaler.fit_transform(combined), index=combined.index, columns=combined.columns)

categories = list(scaled.columns)
fig = go.Figure()
for manager in scaled.index:
    vals = scaled.loc[manager].tolist()
    vals += [vals[0]]  # close the polygon
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=categories + [categories[0]],
        fill='toself', name=manager
    ))
fig.update_layout(polar=dict(radialaxis=dict(visible=False, range=[0, 1])),
                  title=f"Manager Stat Profile — {selected_year}")
st.plotly_chart(fig, use_container_width=True)

In [ ]:
sorted(luck['season'].unique()) + ['all']

[np.int64(2021),
 np.int64(2022),
 np.int64(2023),
 np.int64(2024),
 np.int64(2025),
 np.int64(2026),
 'all']

In [ ]:
## luck index
def expected_wins(group):
    # Expected wins: = (number of managers you outscored) / (total managers - 1)
    n = len(group)
    group = group.copy()
    group['expected_wins'] = group['team_points'].rank(ascending=True) - 1
    group['expected_wins'] = group['expected_wins'] / (n - 1)
    return group

twp = pd.read_csv("data/team_week_points.csv")
twp = twp.groupby(['season', 'week']).apply(expected_wins).reset_index()

exp = (twp[twp['playoffs'] == 0]
       .groupby(['season', 'manager'])['expected_wins']
       .sum().reset_index())

s = pd.read_csv("data/standings_data.csv")
s = s[s.manager.notnull()]
final = s.groupby('season').apply(lambda x: x[x['week'] == x['week'].max()]).reset_index()
actual = final[['season', 'manager', 'wins']]

luck = exp.merge(actual, on=['season', 'manager'])
luck['luck_index'] = luck['wins'] - luck['expected_wins']
luck['luck_index'] = luck['luck_index'].round(1)

selected_year = st.selectbox("Select year", sorted(luck['season'].unique()) + ['all'])
if selected_year == 'all':
    t = luck.groupby(['manager', 'season'])['luck_index'].sum().reset_index()#.sort_values('luck_index')
    t['total_luck'] = t.groupby('manager')['luck_index'].transform('sum')
    plot_df = t.sort_values(['total_luck', 'season'], ascending=[True, True])
    selected_year = 'All Seasons'
else:
    plot_df = luck[luck['season'] == selected_year].sort_values('luck_index')

fig = px.bar(plot_df, x='manager', y='luck_index',
             color='season',
             labels={'luck_index': 'Actual W − Expected W'},
             title=f"Luck Index — {selected_year}")
fig.add_hline(y=0, line_color='gray', line_dash='dash')
fig.update_coloraxes(showscale=False)
#st.plotly_chart(fig, use_container_width=True)

,manager,season,luck_index,total_luck
32,Howard,2023,2.9,8.7
33,Howard,2024,2.9,8.7
30,Howard,2021,3.1,8.7
31,Howard,2022,1.1,8.7
35,Howard,2026,-1.7,8.7
34,Howard,2025,0.4,8.7
29,Garth,2026,-0.3,5.2
28,Garth,2025,2.2,5.2
27,Garth,2024,1.4,5.2
26,Garth,2023,1.1,5.2


In [12]:
#selected_year = st.selectbox("Select year", sorted(luck['season'].unique(), reverse=True))
for year in [2021, 2022, 2023, 2024, 2025, 2026]:
    selected_year = year
    plot_df = luck[luck['season'] == selected_year].sort_values('luck_index')

    fig = px.bar(plot_df, x='manager', y='luck_index',
                color='luck_index', color_continuous_scale='RdYlGn',
                labels={'luck_index': 'Actual W − Expected W'},
                title=f"Luck Index — {selected_year}")
    fig.add_hline(y=0, line_color='gray', line_dash='dash')
    fig.update_coloraxes(showscale=False)
    fig.show()

In [4]:
import numpy as np

In [5]:
from utils import style_pts

In [7]:
SEASON = 2026
pts_select = 'total'
if pts_select == 'total':
    pts_col = 'team_points'
    label = 'Team Points'
elif pts_select == 'hitting':
    pts_col = 'hitting_points'
    label = 'Hitting Points'
else:
    pts_col = 'pitching_points'
    label = 'Pitching Points'

tmp = df.copy()
tmp['points_week_rank'] = tmp.groupby(['season', 'week'])[pts_col].rank(ascending=False, method='min')
current_season = tmp[tmp['season'] == SEASON].copy()
cols_to_color = list(current_season[current_season['week_type'] == 'normal']['week'].unique())
weekly_points = pd.pivot_table(current_season, index=['manager'], columns=['week'], values=pts_col).reset_index()
pts_table = style_pts(weekly_points, False, cols_to_color)


In [11]:
weekly_points = pd.pivot_table(current_season,
                                index=['manager'],
                                columns=['week'],
                                values='points_week_rank').reset_index()
for col in weekly_points.columns:
    if col != 'manager':
        weekly_points[col] = weekly_points[col].astype(int)


In [2]:
df = pd.read_csv("data/team_week_points.csv")

In [3]:
SEASON = 2026
WEEK = df[df['season'] == SEASON]['week'].max()
WEEK_TYPE = df[(df['season'] == SEASON) & (df['week'] == WEEK)]['week_type'].iloc[0]
tmp = df[df['week_type'] == WEEK_TYPE]


In [ ]:

# Overlay highlights in red
fig.add_scatter(
    x=tmp.loc[highlight_mask, '_dummy_x'],
    y=tmp.loc[highlight_mask, 'margin_pct'],
    mode='markers',
    marker=dict(color='red', size=12, symbol='star'),
    hovertemplate='Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>Margin %: %{y:.3f}<extra></extra>',
    customdata=tmp.loc[highlight_mask, ['season', 'week']].values,

)
fig.update_xaxes(range=[-0.2, 0.2], showticklabels=False, title_text='')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
hist_top20 = (player_top
    .query("rank <= 20")
    .groupby(['player_type', 'manager_name', 'season', 'week'])
    .size().reset_index(name='count')
)
for player_type in hist_top20['player_type'].unique():
    print(f"Top 20 {player_type}s by manager:")
    plt.hist(hist_top20[hist_top20['player_type'] == player_type]['count'], bins=range(0, 21), align='left')
    plt.xlim(0, 10)
    plt.show()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

SEASON = 2025
WEEK = 2
tmp = hit.copy()
highlight_mask = (tmp['season'] == SEASON) & (tmp['week'] == WEEK)
tmp['_dummy_x'] = 0

for col in ['R', 'RBI', 'R', 'SB', 'BB', 'HBP', '1B', '2B', '3B', 'HR', 'TB']:
    fig = px.strip(tmp,
                x='_dummy_x',
                y=col,
                color_discrete_sequence=['gray'],
                hover_data={'season': True, 'week': True, col: True, '_dummy_x': False},  # Only show what you want
                )
    fig.data[0].hovertemplate = 'Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>' + col + ': %{y:.3f}<extra></extra>'
    fig.data[0].customdata = tmp[['season', 'week']].values

    # Overlay highlights in red
    fig.add_scatter(
        x=tmp.loc[highlight_mask, '_dummy_x'],
        y=tmp.loc[highlight_mask, col],
        mode='markers',
        marker=dict(color='red', size=12, symbol='star'),
        hovertemplate='Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>' + col + ': %{y:.3f}<extra></extra>',
        customdata=tmp.loc[highlight_mask, ['season', 'week']].values,

    )
    fig.update_xaxes(range=[-0.2, 0.2], showticklabels=False, title_text='')
    fig.update_layout(showlegend=False)
    fig.show()